# Prétraitement de la base E4 : construction de la base nettoyée

Ce notebook définit une fonction de prétraitement `load_and_clean(path)` pour 
l'échantillon 4 (professionnel·les de santé).

Objectifs :
- charger la base brute `Sample 4.csv` ;
- supprimer les colonnes techniques du questionnaire (date, langue, etc.) ;
- regrouper et nettoyer plusieurs variables :
  - activité professionnelle,
  - type de structure d'exercice,
  - tranches d'âge des personnes accompagnées,
  - diagnostics psychiatriques majoritaires ;
- créer une variable `Trouble_Catégorisé` composée de grandes catégories
  diagnostiques (Depression, Anxiety, Bipolar disorder, etc.).

La base nettoyée produite par `load_and_clean` est ensuite réutilisée dans les 
notebooks d'analyse de l'échantillon 4, notamment `BERTopic E4.ipynb`.

In [ ]:
import pandas as pd
import numpy as np
import unicodedata



def load_and_clean(path):
    """
    Charge et nettoie la base de l'échantillon 4 (professionnel·les de santé).

    Étapes principales :
    - lecture du fichier CSV brut ;
    - suppression des colonnes techniques (date de soumission, dernière page,
      langue, consentement, tête de série) ;
    - regroupement des colonnes d'activité professionnelle en une variable
      synthétique (par ex. `Activité professionnelle`) ;
    - création d'une variable `Type de structure` à partir des items cochés
      (hôpital, libéral, etc.) ;
    - création d'une variable `Tranche d'age Accompagnement` à partir des
      tranches d'âge cochées pour les personnes accompagnées ;
    - classification des troubles psychiatriques majoritaires déclarés en
      grandes catégories (Depression, Anxiety, Bipolar disorder, Addiction
      disorder, Personality disorder, Eating disorder, Cognitive disorder,
      Psychotic disorder, Other psychiatric disorder) ;
    - création de la variable `Trouble_Catégorisé` à partir du texte libre.

    Paramètres
    ----------
    path : str
        Chemin vers le fichier CSV brut (par ex. "Data/Sample 4.csv").

    Retour
    ------
    df : pandas.DataFrame
        Base nettoyée et enrichie, prête pour les analyses descriptives et la
        modélisation thématique (BERTopic).
    """
    df = pd.read_csv(path, sep = ';')

    # --- Suppression des colonnes techniques du questionnaire ---
    df.drop(columns=['Date de soumission'], inplace=True)
    df.drop(columns=['Dernière page'], inplace=True)
    df.drop(columns=['Langue de départ'], inplace=True)
    df.drop(columns=["J'accepte"], inplace=True)
    df.drop(columns=["Tête de série"], inplace=True)

    # --- Regroupement de l'activité professionnelle ---
    # (colonnes col_repondant, col_repondant_autre, fonction de fusion,
    #  nouvelle colonne synthétique, puis suppression des colonnes d'origine)


        # Colonnes du répondant
    col_repondant = 'Quelle est votre profession ?'
    col_repondant_autre = 'Quelle est votre profession ? [Autre]'
    
    
    # Fusion des informations répondant
    def fusion_repondant(row):
        std = str(row[col_repondant]).strip() if pd.notna(row[col_repondant]) else ""
        autre = str(row[col_repondant_autre]).strip() if pd.notna(row[col_repondant_autre]) else ""
        if std and autre:
            return f"{std}, {autre}"
        return std or autre or "Non renseigné"
    
    
    # Création des colonnes fusionnées
    df["Activité professionnelle"] = df.apply(fusion_repondant, axis=1)
    
    # Suppression des colonnes d'origine
    df.drop(columns=[
        col_repondant, col_repondant_autre,
    ], inplace=True)



    import re

    # --- Construction de la variable "Type de structure" ---
    # colonnes_structure = [col pour la question "Dans quel type de structure exercez-vous ?"]
    # get_type_structure(row) -> renvoie la structure cochée ou "Autre"
    # création de "Type de structure" et "Type de structure [Autre]"
    # + drop des colonnes d'origine

    # Identifier les colonnes liées à la question
    colonnes_structure = [col for col in df.columns if "Dans quel type de strucutre exercez-vous?" in col]
    colonnes_structure_sans_autre = [col for col in colonnes_structure if "[Autre]" not in col]
    
    # Fonction pour extraire le type sélectionné
    def get_type_structure(row):
        for col in colonnes_structure:
            if row[col] == "Oui":  # Adapter si True / 1
                match = re.search(r"\[(.*?)\]", col)
                if match:
                    return match.group(1)
        # Si aucune colonne cochée
        return "Autre"
    
    # Appliquer la fonction
    df["Type de structure"] = df.apply(get_type_structure, axis=1)
    df["Type de structure [Autre]"] = df["Dans quel type de strucutre exercez-vous? [Autre]"]
    df.drop(columns=colonnes_structure
    , inplace=True)


    # --- Construction de la variable "Tranche d'age Accompagnement" ---
    # col_tranche = colonnes des tranches d'âge cochées ;
    # get_tranche(row) -> renvoie la ou les tranches d'âge pertinentes ;
    # création de "Tranche d'age Accompagnement"
    # + drop des colonnes d'origine
    
    col_tranche = [col for col in df.columns if "Quelle est la tranche d'âge des personnes que vous accompagnez ?" in col] 
    def get_tranche(row):
        for col in col_tranche:
            if row[col] == "Oui":  # Adapter si True / 1
                match = re.search(r"\[(.*?)\]", col)
                if match:
                    return match.group(1)
        return None
    
    # Appliquer la fonction
    df["Tranche d'age Accompagnement"] = df.apply(get_tranche, axis=1)
    df.drop(columns=col_tranche
    , inplace=True)

    # --- Classification des troubles psychiatriques majoritaires ---
    # col_diag = colonne contenant le texte libre sur les troubles psychiatriques majoritaires
    # remove_accents(text) -> supprime les accents
    # classer_troubles(texte) -> renvoie une ou plusieurs catégories cliniques

    col_diag = [col for col in df.columns if "psychiatriques majoritaires" in col]
    
    # Fonction pour supprimer les accents
    def remove_accents(text):
        return ''.join(
            c for c in unicodedata.normalize('NFD', text)
            if unicodedata.category(c) != 'Mn'
        )
    
    # Fonction de classification
    def classer_troubles(texte):
        if pd.isna(texte):
            return np.nan
    
        # Nettoyage du texte
        texte = texte.lower()
        texte = remove_accents(texte)
    
        categories = set()
    
        if any(mot in texte for mot in ["depression", "depressive", "deprime", "depressif", "ts", "edc"]):
            categories.add("Depression")
    
        if any(mot in texte for mot in ["anxiete", "angoisse", "anxiete generalisee", "stress", "anxieux", "tag"]):
            categories.add("Anxiety")
    
        if any(mot in texte for mot in ["bipolaire","bipolarite", "bi polarité", "bi polaire", "bi-polarité", "pmd"]):
            categories.add("Bipolar disorder")
    
        if any(mot in texte for mot in ["alcool", "alcoolisme", "alcoolique", "addiction", "addictions", "dependance", "drogue", "toxicomanie",
                                        "tabac", "tabagique"
                                       ]):
            categories.add("Addiction disorder")
    
        if any(mot in texte for mot in ["borderline", "personnalite", "manipule", "manipulateur", "trouble de perso"]):
            categories.add("Personality disorder")
    
        if any(mot in texte for mot in ["boulimie", "anorexie", "alimentaire", "alimentation", "tca"]):
            categories.add("Eating disorder")
    
        if any(mot in texte for mot in ["alzheimer", "declin", "memoire", "demence", "cognitif", "desorientation", "confusion",
                                       "parkinson", "ecriture et lecture lentes"]):
            categories.add("Cognitive disorder")
    
        if any(mot in texte for mot in [
            "schizophrenie", "schizophrène", "hallucination", "hallucinations", 
            "delire", "delires", "paranoia", "paranoiaque","paranoïde", "psychose", 
            "psychotique", "trouble psychotique", "dissociation", "dissociatif"
        ]):
            categories.add("Psychotic disorder")
    
        # Cas spéciaux ou flous
        if any(mot in texte for mot in [
            "sspt", "ptsd", "trauma", "toc", "hyperactivite", "insecurite", "trouble", "instabilite", 
            "dyspraxie", "hypersensibilite", "trouble obsessionnel compulsif", "lassitude", "burn out",
            "burnout"
        ]):
            categories.add("Other psychiatric disorder")
    
        if not categories:
            return "Other psychiatric disorder"
    
        return ", ".join(sorted(categories))
    
    
    # Application sur une colonne de texte libre
    df["Trouble_Catégorisé"] = df[col_diag[0]].apply(classer_troubles)

    return df


## 2. Application du prétraitement et vérifications

Nous appliquons la fonction `load_and_clean` à la base brute `Sample 4.csv` 
pour obtenir la base nettoyée `df`.  

Nous vérifions ensuite rapidement le résultat.

In [ ]:
# Application de la fonction de prétraitement à la base brute
df = load_and_clean("Data/Sample 4.csv")

# Aperçu des premières lignes de la base nettoyée
df.head()

,ID de la réponse,Genre,Genre [Autre],Age,Quel est votre niveau d'étude?,"Depuis combien d'années exercez-vous ce métier (si vous l'exercez depuis moins d'un an, indiquez 0)?",Inscrivez ici le ou les diagnostic(s) psychiatriques majoritaires que présentent les personnes que vous accompagnez.,"Au quotidien, quels sont les comportements (ex : des crises de boulimies) que vos patients ont qui peuvent les faire souffrir, pouvez-vous nous les expliquer ?","Au quotidien, quelles sont les émotions (ex : tristesse ») qui peuvent éventuellement faire souffrir vos patients pouvez-vous nous les expliquer ?","Au quotidien, quelles sont les pensées (ex : « Je suis nul(le) ») qui peuvent éventuellement faire souffrir vos patients, pouvez-vous nous les expliquer ?",...,Qu’est-ce que vos patients aimeraient pouvoir changer dans leur quotidien ?,"Si vos patients ont suivi une psychothérapie, qu’est-ce que cela leur a apporté ?","Si vos patients ont suivi une psychothérapie, qu’est ce qui a été travaillé avec le thérapeute ?",Qu’est-ce qui peut constituer un frein à l’amélioration de l’état de vos patients ?,Commentaire libre,Activité professionnelle,Type de structure,Type de structure [Autre],Tranche d'age Accompagnement,Trouble_Catégorisé
0,1,femme,NaN,27.0,Bac+5,1.0,Dépression,Test,Test,Test,...,Test,Test,Test,Test,NaN,Infirmier,Service hospitalier de psychiatrie,NaN,Adultes,Depression
1,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Non renseigné,Autre,NaN,None,NaN
2,3,femme,NaN,27.0,Bac+5,NaN,Test,Test,Test,Test,...,Test,Test,Test,Test,NaN,Psychologue,Autre service hospitalier,NaN,Adolescents,Other psychiatric disorder
3,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Non renseigné,Autre,NaN,None,NaN
4,5,femme,NaN,28.0,Bac+3,4.0,"Sevrage alcool, syndrome anxio-depréssif, trou...",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Infirmier,Autre,Clinique privée en psychiatrie,Adultes,"Addiction disorder, Depression, Other psychiat..."


In [ ]:
# Vérification de la taille et des colonnes
df.shape, df.columns

Nous vérifions le nombre de lignes / colonnes et la liste des variables
présentes dans la base nettoyée.

## 3. Sauvegarde de la base nettoyée

La base nettoyée `df` est sauvegardée pour être réutilisée dans les notebooks
d'analyse de l'échantillon 4 (par ex. `BERTopic E4.ipynb`).

In [ ]:
# Sauvegarde de la base nettoyée pour réutilisation ultérieure
df.to_csv("Data/Sample4_clean.csv", sep=",", index=False)